<a href="https://colab.research.google.com/github/niranjanappaji/LLM_Engineering/blob/main/4_LLM_Engg_Brochure_Generator_using_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


AI-Powered Company Brochure Generator & Interactive Chatbot
==========================================================
This script demonstrates an autonomous LLM engineering pipeline that scrapes a company's
landing page, extracts and filters relevant sub-links via structured JSON outputs,
aggregates multi-page content, and streams a professionally formatted company brochure
directly to an interactive Gradio web interface.

In [ ]:
# ==============================================================================
# [Code] Setup Environment & Dependencies
# ==============================================================================
# Install runtime dependencies when this file is executed in Google Colab/Jupyter.

!pip install -q openai bs4 selenium requests

import os
from dotenv import load_dotenv
from openai import OpenAI
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from IPython.display import display, Markdown, update_display
import re
import json
import gradio as gr
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 6.5 MB/s eta 0:00:00


In [ ]:
# ==============================================================================
# [Code] Initialize and Constants
# ==============================================================================

""" Initializes and returns the OpenAI client securely by checking Google Colab
    secrets first, then falling back to standard local environment variables.
"""

def initialize_openai_client() -> OpenAI:

    try:
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = os.environ.get("OPENAI_API_KEY")

    if not api_key:
        raise ValueError("OpenAI API key not found. Please set it in Colab secrets or environment variables.")

    os.environ["OPENAI_API_KEY"] = api_key.strip()
    return OpenAI()


# Constants
OPENAI_CLIENT = initialize_openai_client()
MODEL_NAME = "gpt-4.1-mini"
DEFAULT_SYSTEM_MESSAGE = "You are a helpful and professional AI assistant."


In [ ]:

# ==============================================================================
# 3. Web Scraper & Context Hygiene Utilities
# ==============================================================================

"""
Fetches website HTML via HTTP GET, parses it using BeautifulSoup, decomposes
noisy non-content tags (scripts, styles, footers, navigation), and extracts
cleaned plain text alongside a list of hyperlinks.
"""

def fetch_website_contents(url: str):

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Compatible; BrochureBot/1.0)"},
            timeout=10
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        links = [link.get("href") for link in soup.find_all("a", href=True)]
        links = [link for link in links if link]

        for element in soup(["script", "style", "img", "input", "nav", "footer", "header"]):
            element.decompose()

        text = soup.get_text(" ", strip=True)
        cleaned_text = re.sub(r"\s+", " ", text)
        cleaned_text = re.sub(r"http\S+", " ", cleaned_text).strip()

        if len(cleaned_text) < 100:
            return None

        return {
            "text": cleaned_text,
            "links": links
        }

    except requests.RequestException as e:
        print(f"Network error while fetching {url}: {e}")
        return None
    except Exception as e:
        print(f"Parsing error for {url}: {e}")
        return None

In [ ]:

# ==============================================================================
# 4. Prompt Engineering & Router Configuration
# ==============================================================================

LINK_SELECTOR_SYSTEM_PROMPT = """
You are provided with a list of links found on a website.
You must decide which of the links are most relevant to include in a company brochure,
such as links to an 'About' page, company overview, product pages, or careers page.
You must respond strictly in valid JSON matching the following schema structure:

{
  "links": [
    {"type": "about page", "url": "https://example.com/about"},
    {"type": "products page", "url": "https://example.com/products"}
  ]
}
"""


# Constructs the user prompt containing raw scraped links for AI routing.

def build_link_selection_prompt(url: str):

    scraped_data = fetch_website_contents(url)
    if not scraped_data or not scraped_data["links"]:
        return f"Here is the list of links found on a website {url}:\n"

    user_prompt = (
        f"Here is the list of links found on a website {url} -\n"
        "Please decide which links are most relevant for a company brochure in JSON format.\n"
        "Do not include Terms of Service, Privacy policies, or email/mailto links.\n\n"
        "Links:\n"
    )
    user_prompt += "\n".join(scraped_data["links"])
    return user_prompt


def select_relevant_links(url: str):
    """
    Acts as an intelligent router: passes scraped links to the LLM under strict
    JSON formatting rules to isolate high-value business sub-pages.
    """
    print(f"Selecting relevant links for {url} using model {MODEL_NAME}...")

    try:
        response = OPENAI_CLIENT.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": LINK_SELECTOR_SYSTEM_PROMPT},
                {"role": "user", "content": build_link_selection_prompt(url)}
            ],
            response_format={"type": "json_object"}
        )

        result_content = response.choices[0].message.content
        parsed_links = json.loads(result_content)
        print(f"Successfully identified {len(parsed_links.get('links', []))} relevant links.")
        return parsed_links

    except Exception as e:
        print(f"Error during link selection routing: {e}")
        return {"links": []}


In [ ]:

# ==============================================================================
# 5. Multi-Page Context Aggregation & Brochure Generation
# ==============================================================================

"""
    Orchestrates multi-page aggregation, scrapes the landing page, routes and filters
    sub-links via LLM, deep-scrapes each target page, and stitches them into a unified context corpus.
"""

def fetch_text_and_relevant_links(url: str):

    print(f"Aggregating content and sub-pages for: {url}")

    landing_result = fetch_website_contents(url)
    if not landing_result:
        return None

    landing_text = landing_result["text"]
    relevant_links_data = select_relevant_links(url)

    aggregated_corpus = f"## Landing Page:\n\n{landing_text}\n\n## Relevant Sub-Pages:\n"

    for link_item in relevant_links_data.get("links", []):
        sub_url = link_item.get("url")
        link_type = link_item.get("type", "page")

        sub_result = fetch_website_contents(sub_url)
        if sub_result is None:
            continue

        aggregated_corpus += f"\n\n### Link Type: {link_type} ({sub_url})\n"
        aggregated_corpus += sub_result["text"]

    return aggregated_corpus


# Streams token responses iteratively from the OpenAI chat completion endpoint.


def stream_gpt(prompt: str):

    stream = OPENAI_CLIENT.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": DEFAULT_SYSTEM_MESSAGE},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )

    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        yield content

"""
    Main generator workflow: compiles multi-page web context and streams a professional
    Markdown company brochure chunk-by-chunk.
"""

def stream_brochure(company_name: str, url: str):

    yield "Initializing scraping and multi-page routing pipeline..."

    page_data_corpus = fetch_text_and_relevant_links(url)
    if not page_data_corpus:
        yield "Error: Could not retrieve or parse website contents from the provided URL."
        return

    brochure_system_prompt = (
        "You are an expert corporate communications assistant that analyzes website contents "
        "and builds comprehensive company brochures for customers, investors, and prospective recruits. "
        "Respond in professional markdown without code blocks. "
        "Include details regarding company culture, core products, customer feedback, and careers if available."
    )

    user_prompt = (
        f"Generate a professional company brochure for '{company_name}'. "
        f"Here is the aggregated content from their landing page and relevant sub-pages:\n\n"
        f"{page_data_corpus}"
    )

    stream = OPENAI_CLIENT.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    response_accumulator = ""
    for chunk in stream:
        response_accumulator += chunk.choices[0].delta.content or ""
        yield response_accumulator



In [ ]:

# ==============================================================================
# 6. User Interface (Gradio Web App)
# ==============================================================================

if __name__ == "__main__":
    # Define Gradio interface components
    name_input = gr.Textbox(
        label="Company Name",
        placeholder="e.g., Pine Cone Home Decor"
    )
    url_input = gr.Textbox(
        label="Landing Page URL",
        placeholder="https://example.com"
    )
    output_markdown = gr.Markdown(
        label="Generated Company Brochure"
    )

    # Launch public web interface with automatic browser popup and basic auth
    app = gr.Interface(
        fn=stream_brochure,
        title="AI-Powered Company Brochure Generator",
        description="Enter a company name and landing page URL to autonomously scrape, route, and stream a custom brochure.",
        inputs=[name_input, url_input],
        outputs=output_markdown,
        flagging_mode="never",
        examples=[
            ["Adukale", "https://us.adukale.com/"],
            ["Pine Cone India", "https://pineconeindia.in/"]
        ]
    )

    app.launch(share=True, inbrowser=True, auth=("username", "password"))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f28ed13ce4f9ae6a8b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
